# (Optional) S3 Tables 생성 및 데이터 마이그레이션

Aurora PostgreSQL → S3 Tables (`data` namespace) 마이그레이션

**VS Code에서 실행합니다. EMR Serverless Livy를 직접 호출합니다.**

## Steps
1. EMR Serverless Livy 세션 초기화
2. S3 Tables에 24개 테이블 생성
3. Aurora PostgreSQL → S3 Tables 데이터 로딩
4. 검증

## Step 1: Setup — EMR Serverless Livy 세션 초기화

In [ ]:
import boto3, json, time, os, botocore.session
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

session = boto3.session.Session()
REGION = session.region_name
ACCOUNT_ID = session.client('sts').get_caller_identity()['Account']

# EMR Serverless app 자동 탐색
emr_client = boto3.client('emr-serverless', region_name=REGION)
apps = emr_client.list_applications()['applications']
emr_app = next(a for a in apps if a['name'] == 'fhir-data-query')
APPLICATION_ID = emr_app['id']

iam_client = boto3.client('iam')
roles = iam_client.list_roles(MaxItems=200)['Roles']
exec_role = next(r for r in roles if 'emr-serverless-execution-role' in r['RoleName'])
EXECUTION_ROLE_ARN = exec_role['Arn']

ENDPOINT = f"https://{APPLICATION_ID}.livy.emr-serverless-services.{REGION}.amazonaws.com"
HEADERS = {"Content-Type": "application/json"}

print(f"EMR Application: {APPLICATION_ID}")
print(f"Execution Role: {EXECUTION_ROLE_ARN}")

def _sign(method, url, data=None):
    bs = botocore.session.Session()
    signer = SigV4Auth(bs.get_credentials(), "emr-serverless", REGION)
    body = json.dumps(data) if data else None
    req = AWSRequest(method=method, url=url, data=body, headers=HEADERS)
    signer.add_auth(req)
    p = req.prepare()
    fn = getattr(requests, method.lower())
    return fn(p.url, headers=p.headers, data=body)

# Create/reuse Livy session
_session_id = None

def get_session():
    global _session_id
    if _session_id:
        r = _sign("GET", f"{ENDPOINT}/sessions/{_session_id}/state")
        if r.status_code == 200 and r.json().get('state') in ('idle', 'busy'):
            print(f"Reusing session: {_session_id}")
            return _session_id

    secret_id = boto3.client('secretsmanager', region_name=REGION).list_secrets()['SecretList'][0]['Name']
    creds_str = boto3.client('secretsmanager', region_name=REGION).get_secret_value(SecretId=secret_id)['SecretString']
    creds = json.loads(creds_str)
    jdbc_url = f"jdbc:postgresql://{creds['host']}:{creds['port']}/{creds['dbname']}"
    driver_path = f"s3://fhir-data-{ACCOUNT_ID}-{REGION}/lib/postgresql-42.7.1.jar"

    conf = {
        "kind": "sql",
        "conf": {
            "spark.jars": driver_path,
            "spark.driver.extraClassPath": driver_path,
            "spark.executor.extraClassPath": driver_path,
            f"spark.sql.catalog.s3tablescatalog_fhir-bucket": "org.apache.iceberg.spark.SparkCatalog",
            f"spark.sql.catalog.s3tablescatalog_fhir-bucket.catalog-impl": "software.amazon.s3tables.iceberg.S3TablesCatalog",
            f"spark.sql.catalog.s3tablescatalog_fhir-bucket.warehouse": f"arn:aws:s3tables:{REGION}:{ACCOUNT_ID}:bucket/fhir-bucket",
        },
        "executorMemory": "8g",
        "executorCores": 4,
        "numExecutors": 4,
    }
    # Store JDBC info for later use
    import builtins
    builtins._jdbc_url = jdbc_url
    builtins._jdbc_creds = creds
    builtins._driver_path = driver_path

    r = _sign("POST", f"{ENDPOINT}/sessions", conf)
    _session_id = r.json()['id']
    print(f"Creating session {_session_id}...")
    for _ in range(60):
        state = _sign("GET", f"{ENDPOINT}/sessions/{_session_id}/state").json().get('state')
        if state == 'idle': break
        print(f"  {state}", end='\r')
        time.sleep(5)
    print(f"Session ready: {_session_id}")
    return _session_id

def run_sql(sql, session_id=None):
    sid = session_id or get_session()
    r = _sign("POST", f"{ENDPOINT}/sessions/{sid}/statements", {"code": sql, "kind": "sql"})
    stmt_id = r.json()['id']
    for _ in range(120):
        r = _sign("GET", f"{ENDPOINT}/sessions/{sid}/statements/{stmt_id}")
        s = r.json()
        if s['state'] == 'available':
            out = s.get('output', {})
            if out.get('status') == 'error':
                print(f"ERROR: {out.get('evalue', '')}")
                return None
            return out.get('data', {})
        time.sleep(3)
    print("Timeout")
    return None

def run_pyspark(code_str, session_id=None):
    """Run PySpark code via Livy"""
    sid = session_id or get_session()
    r = _sign("POST", f"{ENDPOINT}/sessions/{sid}/statements", {"code": code_str, "kind": "pyspark"})
    stmt_id = r.json()['id']
    for _ in range(120):
        r = _sign("GET", f"{ENDPOINT}/sessions/{sid}/statements/{stmt_id}")
        s = r.json()
        if s['state'] == 'available':
            out = s.get('output', {})
            if out.get('status') == 'error':
                print(f"ERROR: {out.get('evalue', '')}")
                return None
            return out.get('data', {})
        time.sleep(3)
    return None

SESSION_ID = get_session()
print(f"\nReady. Session ID: {SESSION_ID}")


## Step 2: S3 Tables 테이블 생성

In [ ]:
run_sql("USE `s3tablescatalog_fhir-bucket`.data", SESSION_ID)
print("Using s3tablescatalog_fhir-bucket.data")


# FHIR S3 Tables Creation

Creates 24 tables with 498 columns from FHIR metadata.

Domains: **Administrative** (5), **Clinical** (12), **Medication** (3), **Financial** (2), **Security** (2)

In [ ]:
# === OPTIONAL: Drop all tables (uncomment to use) ===
# run_sql("DROP TABLE IF EXISTS patient")
# run_sql("DROP TABLE IF EXISTS practitioner")
# run_sql("DROP TABLE IF EXISTS organization")
# run_sql("DROP TABLE IF EXISTS location")
# run_sql("DROP TABLE IF EXISTS practitioner_role")
# run_sql("DROP TABLE IF EXISTS encounter")
# run_sql("DROP TABLE IF EXISTS condition")
# run_sql("DROP TABLE IF EXISTS observation")
# run_sql("DROP TABLE IF EXISTS procedure")
# run_sql("DROP TABLE IF EXISTS allergy_intolerance")
# run_sql("DROP TABLE IF EXISTS diagnostic_report")
# run_sql("DROP TABLE IF EXISTS imaging_study")
# run_sql("DROP TABLE IF EXISTS immunization")
# run_sql("DROP TABLE IF EXISTS medication_administration")
# run_sql("DROP TABLE IF EXISTS medication")
# run_sql("DROP TABLE IF EXISTS medication_request")
# run_sql("DROP TABLE IF EXISTS claim")
# run_sql("DROP TABLE IF EXISTS explanation_of_benefit")
# run_sql("DROP TABLE IF EXISTS care_plan")
# run_sql("DROP TABLE IF EXISTS care_team")
# run_sql("DROP TABLE IF EXISTS device")
# run_sql("DROP TABLE IF EXISTS supply_delivery")
# run_sql("DROP TABLE IF EXISTS document_reference")
# run_sql("DROP TABLE IF EXISTS provenance")
# print("All tables dropped.")

## Administrative Domain (5 tables)

Tables: `patient`, `practitioner`, `organization`, `location`, `practitioner_role`

In [ ]:
# Patient - Patient demographic and administrative information including name, gender, birth
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS patient (
        address_city STRING COMMENT 'City name',
        multiple_birth_indicator BOOLEAN COMMENT 'Whether patient is part of a multiple birth',
        gender STRING COMMENT 'Administrative gender',
        telecom_system STRING COMMENT 'Telecom system type (phone, email, etc.)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        address_postal_code STRING COMMENT 'Postal/ZIP code',
        address_country STRING COMMENT 'Country code',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        deceased_datetime TIMESTAMP COMMENT 'Date/time of death',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        language_code STRING COMMENT 'Preferred language code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        name_family STRING COMMENT 'Family (last) name',
        telecom_value STRING COMMENT 'Telecom contact value',
        birth_date DATE COMMENT 'Date of birth',
        identifier_value STRING COMMENT 'Identifier value within the system',
        address_line_1 STRING COMMENT 'Street address line 1',
        address_state STRING COMMENT 'State or province',
        multiple_birth_number INT COMMENT 'Birth order in multiple birth',
        address_line_2 STRING COMMENT 'Street address line 2',
        marital_status STRING COMMENT 'Marital status',
        name_given STRING COMMENT 'Given (first) name'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Patient',
        'domain' = 'Administrative',
        'sub_domain' = 'Demographics',
        'description' = 'Patient demographic and administrative information including name, gender, birth date, address, and contact details'
    )
    """, SESSION_ID)
    print("Created table: patient (24 columns)")
except Exception as e:
    print(f"Error creating patient: {e}")

In [ ]:
# Practitioner - Healthcare practitioner information including name, qualifications, gender, and 
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS practitioner (
        address_city STRING COMMENT 'City name',
        name_prefix STRING COMMENT 'Name prefix (e.g., Dr., Mr.)',
        gender STRING COMMENT 'Administrative gender',
        name_suffix STRING COMMENT 'Name suffix (e.g., Jr., III)',
        telecom_system STRING COMMENT 'Telecom system type (phone, email, etc.)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        address_postal_code STRING COMMENT 'Postal/ZIP code',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        name_family STRING COMMENT 'Family (last) name',
        telecom_value STRING COMMENT 'Telecom contact value',
        birth_date DATE COMMENT 'Date of birth',
        identifier_value STRING COMMENT 'Identifier value within the system',
        address_line_1 STRING COMMENT 'Street address line 1',
        address_state STRING COMMENT 'State or province',
        active_indicator BOOLEAN COMMENT 'Whether the resource is currently active',
        name_given STRING COMMENT 'Given (first) name'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Practitioner',
        'domain' = 'Administrative',
        'sub_domain' = 'Personnel',
        'description' = 'Healthcare practitioner information including name, qualifications, gender, and contact details'
    )
    """, SESSION_ID)
    print("Created table: practitioner (19 columns)")
except Exception as e:
    print(f"Error creating practitioner: {e}")

In [ ]:
# Organization - Healthcare organization details including name, type, address, and active status
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS organization (
        address_city STRING COMMENT 'City name',
        telecom_system STRING COMMENT 'Telecom system type (phone, email, etc.)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        address_postal_code STRING COMMENT 'Postal/ZIP code',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        telecom_value STRING COMMENT 'Telecom contact value',
        identifier_value STRING COMMENT 'Identifier value within the system',
        address_line_1 STRING COMMENT 'Street address line 1',
        address_state STRING COMMENT 'State or province',
        type_code STRING COMMENT 'Type code value',
        active_indicator BOOLEAN COMMENT 'Whether the resource is currently active',
        name STRING COMMENT 'Name of the resource'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Organization',
        'domain' = 'Administrative',
        'sub_domain' = 'Entities',
        'description' = 'Healthcare organization details including name, type, address, and active status'
    )
    """, SESSION_ID)
    print("Created table: organization (16 columns)")
except Exception as e:
    print(f"Error creating organization: {e}")

In [ ]:
# Location - Physical location or place where healthcare services are provided, including add
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS location (
        address_city STRING COMMENT 'City name',
        managing_organization_reference STRING COMMENT 'Reference to the managing organization',
        telecom_system STRING COMMENT 'Telecom system type (phone, email, etc.)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        address_postal_code STRING COMMENT 'Postal/ZIP code',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        position_latitude DECIMAL(10,7) COMMENT 'Geographic latitude coordinate',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        telecom_value STRING COMMENT 'Telecom contact value',
        description STRING COMMENT 'Free-text description',
        identifier_value STRING COMMENT 'Identifier value within the system',
        address_line_1 STRING COMMENT 'Street address line 1',
        address_state STRING COMMENT 'State or province',
        mode STRING COMMENT 'Location mode (instance, kind)',
        type_code STRING COMMENT 'Type code value',
        name STRING COMMENT 'Name of the resource',
        position_longitude DECIMAL(10,7) COMMENT 'Geographic longitude coordinate'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Location',
        'domain' = 'Administrative',
        'sub_domain' = 'Facilities',
        'description' = 'Physical location or place where healthcare services are provided, including address and geo-coordinates'
    )
    """, SESSION_ID)
    print("Created table: location (21 columns)")
except Exception as e:
    print(f"Error creating location: {e}")

In [ ]:
# PractitionerRole - Roles and specialties of practitioners within organizations at specific location
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS practitioner_role (
        specialty_value STRING COMMENT 'Practitioner specialty code value',
        specialty_display STRING COMMENT 'Practitioner specialty display text',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        organization_reference STRING COMMENT 'Reference to the organization',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        code_system STRING COMMENT 'Code system URI for the primary code',
        practitioner_reference STRING COMMENT 'Reference to the practitioner',
        identifier_value STRING COMMENT 'Identifier value within the system',
        active_indicator BOOLEAN COMMENT 'Whether the resource is currently active',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code',
        location_reference STRING COMMENT 'Reference to the location',
        specialty_system STRING COMMENT 'Practitioner specialty code system URI'
    ) TBLPROPERTIES (
        'fhir_resource' = 'PractitionerRole',
        'domain' = 'Administrative',
        'sub_domain' = 'Personnel',
        'description' = 'Roles and specialties of practitioners within organizations at specific locations'
    )
    """, SESSION_ID)
    print("Created table: practitioner_role (16 columns)")
except Exception as e:
    print(f"Error creating practitioner_role: {e}")

## Clinical Domain (12 tables)

Tables: `encounter`, `condition`, `observation`, `procedure`, `allergy_intolerance`, `diagnostic_report`, `imaging_study`, `immunization`, `care_plan`, `care_team`, `device`, `supply_delivery`

In [ ]:
# Encounter - Patient encounters (visits) with healthcare providers including type, class, per
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS encounter (
        reason_code_system STRING COMMENT 'Reason code system URI',
        class_code STRING COMMENT 'Encounter class code (ambulatory, emergency, inpatient, etc.)',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        period_end_datetime TIMESTAMP COMMENT 'Period end datetime',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        organization_reference STRING COMMENT 'Reference to the organization',
        class_system STRING COMMENT 'Encounter class code system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        period_start_datetime TIMESTAMP COMMENT 'Period start datetime',
        service_provider_reference STRING COMMENT 'Reference to the service provider organization',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        reason_code_value STRING COMMENT 'Reason code value',
        reason_code_display STRING COMMENT 'Reason code display text',
        identifier_value STRING COMMENT 'Identifier value within the system',
        type_code STRING COMMENT 'Type code value',
        type_system STRING COMMENT 'Type code system URI',
        location_reference STRING COMMENT 'Reference to the location'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Encounter',
        'domain' = 'Clinical',
        'sub_domain' = 'Encounters',
        'description' = 'Patient encounters (visits) with healthcare providers including type, class, period, reason, and service provider'
    )
    """, SESSION_ID)
    print("Created table: encounter (21 columns)")
except Exception as e:
    print(f"Error creating encounter: {e}")

In [ ]:
# Condition - Clinical conditions, diagnoses, and health concerns including onset, clinical st
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS condition (
        clinical_status_system STRING COMMENT 'Clinical status code system URI',
        abatement_datetime TIMESTAMP COMMENT 'Abatement (resolution) datetime',
        clinical_status_code STRING COMMENT 'Clinical status code (active, resolved, etc.)',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        category_code STRING COMMENT 'Category code value',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        category_display STRING COMMENT 'Category display text',
        recorded_date DATE COMMENT 'Date the resource was recorded',
        resource_id STRING COMMENT 'Unique resource identifier',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        code_text STRING COMMENT 'Free-text description of the code',
        verification_status_code STRING COMMENT 'Verification status code (confirmed, unconfirmed, etc.)',
        code_system STRING COMMENT 'Code system URI for the primary code',
        onset_datetime TIMESTAMP COMMENT 'Onset datetime of the condition',
        category_system STRING COMMENT 'Category code system URI',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        verification_status_system STRING COMMENT 'Verification status code system URI',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Condition',
        'domain' = 'Clinical',
        'sub_domain' = 'Diagnoses',
        'description' = 'Clinical conditions, diagnoses, and health concerns including onset, clinical status, and verification status'
    )
    """, SESSION_ID)
    print("Created table: condition (20 columns)")
except Exception as e:
    print(f"Error creating condition: {e}")

In [ ]:
# Observation - Clinical observations and measurements including vital signs, lab results, and a
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS observation (
        issued_datetime TIMESTAMP COMMENT 'Date/time the resource was issued',
        interpretation_text STRING COMMENT 'Interpretation or comment text',
        effective_datetime TIMESTAMP COMMENT 'Clinically effective datetime',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        category_code STRING COMMENT 'Category code value',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        category_display STRING COMMENT 'Category display text',
        value_system STRING COMMENT 'Value code system URI',
        resource_id STRING COMMENT 'Unique resource identifier',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        code_text STRING COMMENT 'Free-text description of the code',
        code_system STRING COMMENT 'Code system URI for the primary code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        value_boolean BOOLEAN COMMENT 'Boolean value of the observation',
        category_system STRING COMMENT 'Category code system URI',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        value_string STRING COMMENT 'String value of the observation',
        value_unit STRING COMMENT 'Unit of the quantity value',
        value_code STRING COMMENT 'Coded value',
        value_quantity DECIMAL(18,6) COMMENT 'Numeric quantity value',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Observation',
        'domain' = 'Clinical',
        'sub_domain' = 'Measurements',
        'description' = 'Clinical observations and measurements including vital signs, lab results, and assessments with values and units'
    )
    """, SESSION_ID)
    print("Created table: observation (23 columns)")
except Exception as e:
    print(f"Error creating observation: {e}")

In [ ]:
# Procedure - Clinical procedures performed on patients including type, performer, location, a
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS procedure (
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        code_text STRING COMMENT 'Free-text description of the code',
        performed_start_datetime TIMESTAMP COMMENT 'Procedure performed start datetime',
        code_system STRING COMMENT 'Code system URI for the primary code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        performed_end_datetime TIMESTAMP COMMENT 'Procedure performed end datetime',
        practitioner_reference STRING COMMENT 'Reference to the practitioner',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code',
        location_reference STRING COMMENT 'Reference to the location',
        reason_reference STRING COMMENT 'Reference to the reason condition'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Procedure',
        'domain' = 'Clinical',
        'sub_domain' = 'Procedures',
        'description' = 'Clinical procedures performed on patients including type, performer, location, and reason'
    )
    """, SESSION_ID)
    print("Created table: procedure (16 columns)")
except Exception as e:
    print(f"Error creating procedure: {e}")

In [ ]:
# AllergyIntolerance - Patient allergy and intolerance records including substance, clinical status, cr
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS allergy_intolerance (
        clinical_status_system STRING COMMENT 'Clinical status code system URI',
        clinical_status_code STRING COMMENT 'Clinical status code (active, resolved, etc.)',
        criticality STRING COMMENT 'Allergy criticality level',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        type STRING COMMENT 'Type value',
        recorded_date TIMESTAMP COMMENT 'Date the resource was recorded',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        verification_status_code STRING COMMENT 'Verification status code (confirmed, unconfirmed, etc.)',
        code_system STRING COMMENT 'Code system URI for the primary code',
        onset_datetime TIMESTAMP COMMENT 'Onset datetime of the condition',
        category STRING COMMENT 'Category value',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        identifier_value STRING COMMENT 'Identifier value within the system',
        verification_status_system STRING COMMENT 'Verification status code system URI',
        recorder_reference STRING COMMENT 'Reference to the recorder of the resource',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code'
    ) TBLPROPERTIES (
        'fhir_resource' = 'AllergyIntolerance',
        'domain' = 'Clinical',
        'sub_domain' = 'Allergies',
        'description' = 'Patient allergy and intolerance records including substance, clinical status, criticality, and onset'
    )
    """, SESSION_ID)
    print("Created table: allergy_intolerance (21 columns)")
except Exception as e:
    print(f"Error creating allergy_intolerance: {e}")

In [ ]:
# DiagnosticReport - Diagnostic reports including lab results and imaging findings with status, categ
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS diagnostic_report (
        issued_datetime TIMESTAMP COMMENT 'Date/time the resource was issued',
        effective_datetime TIMESTAMP COMMENT 'Clinically effective datetime',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        category_code STRING COMMENT 'Category code value',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        category_display STRING COMMENT 'Category display text',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        code_text STRING COMMENT 'Free-text description of the code',
        code_system STRING COMMENT 'Code system URI for the primary code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        practitioner_reference STRING COMMENT 'Reference to the practitioner',
        category_system STRING COMMENT 'Category code system URI',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        conclusion_text STRING COMMENT 'Conclusion or summary text',
        identifier_value STRING COMMENT 'Identifier value within the system',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code'
    ) TBLPROPERTIES (
        'fhir_resource' = 'DiagnosticReport',
        'domain' = 'Clinical',
        'sub_domain' = 'Diagnostics',
        'description' = 'Diagnostic reports including lab results and imaging findings with status, category, and conclusions'
    )
    """, SESSION_ID)
    print("Created table: diagnostic_report (20 columns)")
except Exception as e:
    print(f"Error creating diagnostic_report: {e}")

In [ ]:
# ImagingStudy - Imaging studies (X-ray, CT, MRI, etc.) with modality, series/instance counts, an
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS imaging_study (
        reason_code_system STRING COMMENT 'Reason code system URI',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        started_datetime TIMESTAMP COMMENT 'Study started datetime',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        interpreter_reference STRING COMMENT 'Reference to the interpreter',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        modality_system STRING COMMENT 'Imaging modality code system URI',
        procedure_code_system STRING COMMENT 'Procedure code system URI',
        modality_code STRING COMMENT 'Imaging modality code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        number_of_instances INT COMMENT 'Number of imaging instances',
        reason_code_value STRING COMMENT 'Reason code value',
        reason_code_display STRING COMMENT 'Reason code display text',
        description STRING COMMENT 'Free-text description',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        identifier_value STRING COMMENT 'Identifier value within the system',
        referrer_reference STRING COMMENT 'Reference to the referring practitioner',
        number_of_series INT COMMENT 'Number of imaging series',
        modality_display STRING COMMENT 'Imaging modality display text',
        procedure_code_value STRING COMMENT 'Procedure code value',
        procedure_code_display STRING COMMENT 'Procedure code display text'
    ) TBLPROPERTIES (
        'fhir_resource' = 'ImagingStudy',
        'domain' = 'Clinical',
        'sub_domain' = 'Imaging',
        'description' = 'Imaging studies (X-ray, CT, MRI, etc.) with modality, series/instance counts, and procedure codes'
    )
    """, SESSION_ID)
    print("Created table: imaging_study (24 columns)")
except Exception as e:
    print(f"Error creating imaging_study: {e}")

In [ ]:
# Immunization - Immunization/vaccination records including vaccine code, dose, route, site, and 
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS immunization (
        vaccine_code_display STRING COMMENT 'Vaccine code display text',
        status_reason_system STRING COMMENT 'Status reason code system URI',
        lot_number STRING COMMENT 'Manufacturing lot number',
        primary_source BOOLEAN COMMENT 'Whether data is from primary source',
        manufacturer_reference STRING COMMENT 'Reference to the manufacturer organization',
        route_system STRING COMMENT 'Administration route code system URI',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        status_reason_display STRING COMMENT 'Status reason display text',
        occurrence_datetime TIMESTAMP COMMENT 'Date/time of occurrence',
        route_display STRING COMMENT 'Administration route display text',
        site_code STRING COMMENT 'Body site code',
        expiration_date DATE COMMENT 'Expiration date',
        vaccine_code_system STRING COMMENT 'Vaccine code system URI',
        dosage_unit STRING COMMENT 'Dosage unit',
        vaccine_code_value STRING COMMENT 'Vaccine code value',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        dosage_quantity DECIMAL(18,6) COMMENT 'Dosage quantity',
        status_reason_code STRING COMMENT 'Status reason code',
        route_code STRING COMMENT 'Administration route code',
        site_system STRING COMMENT 'Body site code system URI',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        practitioner_reference STRING COMMENT 'Reference to the practitioner',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        site_display STRING COMMENT 'Body site display text',
        location_reference STRING COMMENT 'Reference to the location'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Immunization',
        'domain' = 'Clinical',
        'sub_domain' = 'Immunizations',
        'description' = 'Immunization/vaccination records including vaccine code, dose, route, site, and lot number'
    )
    """, SESSION_ID)
    print("Created table: immunization (28 columns)")
except Exception as e:
    print(f"Error creating immunization: {e}")

In [ ]:
# CarePlan - Patient care plans including intent, category, period, title, and description of
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS care_plan (
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        period_end_datetime TIMESTAMP COMMENT 'Period end datetime',
        category_code STRING COMMENT 'Category code value',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        category_display STRING COMMENT 'Category display text',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        title STRING COMMENT 'Title of the resource',
        intent STRING COMMENT 'Intent of the resource (proposal, plan, order, etc.)',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        period_start_datetime TIMESTAMP COMMENT 'Period start datetime',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        description STRING COMMENT 'Free-text description',
        category_system STRING COMMENT 'Category code system URI',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        identifier_value STRING COMMENT 'Identifier value within the system'
    ) TBLPROPERTIES (
        'fhir_resource' = 'CarePlan',
        'domain' = 'Clinical',
        'sub_domain' = 'Care Plans',
        'description' = 'Patient care plans including intent, category, period, title, and description of planned care activities'
    )
    """, SESSION_ID)
    print("Created table: care_plan (17 columns)")
except Exception as e:
    print(f"Error creating care_plan: {e}")

In [ ]:
# CareTeam - Care team composition including managing organization, category, period, and tea
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS care_team (
        managing_organization_reference STRING COMMENT 'Reference to the managing organization',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        period_end_datetime TIMESTAMP COMMENT 'Period end datetime',
        category_code STRING COMMENT 'Category code value',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        category_display STRING COMMENT 'Category display text',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        period_start_datetime TIMESTAMP COMMENT 'Period start datetime',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        category_system STRING COMMENT 'Category code system URI',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        identifier_value STRING COMMENT 'Identifier value within the system',
        name STRING COMMENT 'Name of the resource'
    ) TBLPROPERTIES (
        'fhir_resource' = 'CareTeam',
        'domain' = 'Clinical',
        'sub_domain' = 'Care Teams',
        'description' = 'Care team composition including managing organization, category, period, and team name'
    )
    """, SESSION_ID)
    print("Created table: care_team (16 columns)")
except Exception as e:
    print(f"Error creating care_team: {e}")

In [ ]:
# Device - Medical devices associated with patients including type, manufacturer, model, se
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS device (
        lot_number STRING COMMENT 'Manufacturing lot number',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        serial_number STRING COMMENT 'Device serial number',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        model_number STRING COMMENT 'Device model number',
        version STRING COMMENT 'Device version',
        identifier_value STRING COMMENT 'Identifier value within the system',
        type_text STRING COMMENT 'Free-text type description',
        manufacturer STRING COMMENT 'Manufacturer name',
        type_code STRING COMMENT 'Type code value',
        expiration_date DATE COMMENT 'Expiration date',
        type_system STRING COMMENT 'Type code system URI',
        location_reference STRING COMMENT 'Reference to the location'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Device',
        'domain' = 'Clinical',
        'sub_domain' = 'Devices',
        'description' = 'Medical devices associated with patients including type, manufacturer, model, serial number, and expiration'
    )
    """, SESSION_ID)
    print("Created table: device (19 columns)")
except Exception as e:
    print(f"Error creating device: {e}")

In [ ]:
# SupplyDelivery - Supply delivery records including supplied item, quantity, destination, supplier
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS supply_delivery (
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        destination_reference STRING COMMENT 'Reference to the delivery destination',
        supplier_reference STRING COMMENT 'Reference to the supplier',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        type_display STRING COMMENT 'Type display text',
        occurrence_end_datetime TIMESTAMP COMMENT 'Occurrence period end datetime',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        occurrence_start_datetime TIMESTAMP COMMENT 'Occurrence period start datetime',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        supplied_item_code_system STRING COMMENT 'Supplied item code system URI',
        quantity DECIMAL(18,6) COMMENT 'Quantity value',
        identifier_value STRING COMMENT 'Identifier value within the system',
        supplied_item_code_value STRING COMMENT 'Supplied item code value',
        supplied_item_code_display STRING COMMENT 'Supplied item code display text',
        type_code STRING COMMENT 'Type code value',
        quantity_unit STRING COMMENT 'Quantity unit',
        type_system STRING COMMENT 'Type code system URI',
        supplied_item_reference STRING COMMENT 'Reference to the supplied item'
    ) TBLPROPERTIES (
        'fhir_resource' = 'SupplyDelivery',
        'domain' = 'Clinical',
        'sub_domain' = 'Supply Chain',
        'description' = 'Supply delivery records including supplied item, quantity, destination, supplier, and occurrence period'
    )
    """, SESSION_ID)
    print("Created table: supply_delivery (21 columns)")
except Exception as e:
    print(f"Error creating supply_delivery: {e}")

## Financial Domain (2 tables)

Tables: `claim`, `explanation_of_benefit`

In [ ]:
# Claim - Insurance claims for healthcare services including total amount, type, priority,
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS claim (
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        insurer_reference STRING COMMENT 'Reference to the insurance organization',
        provider_reference STRING COMMENT 'Reference to the billing provider',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        total_amount DECIMAL(18,2) COMMENT 'Total monetary amount',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        priority_code_system STRING COMMENT 'Priority code system URI',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        use_code STRING COMMENT 'Use code (claim, preauthorization, etc.)',
        priority STRING COMMENT 'Processing priority',
        total_currency STRING COMMENT 'Total amount currency code',
        system_created_datetime TIMESTAMP COMMENT 'System-level creation timestamp',
        identifier_value STRING COMMENT 'Identifier value within the system',
        type_code STRING COMMENT 'Type code value',
        type_system STRING COMMENT 'Type code system URI',
        system_updated_datetime TIMESTAMP COMMENT 'System-level update timestamp',
        priority_code_value STRING COMMENT 'Priority code value',
        priority_code_display STRING COMMENT 'Priority code display text'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Claim',
        'domain' = 'Financial',
        'sub_domain' = 'Claims',
        'description' = 'Insurance claims for healthcare services including total amount, type, priority, and provider references'
    )
    """, SESSION_ID)
    print("Created table: claim (21 columns)")
except Exception as e:
    print(f"Error creating claim: {e}")

In [ ]:
# ExplanationOfBenefit - Explanation of benefit records linking claims to payments including outcome, pay
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS explanation_of_benefit (
        outcome STRING COMMENT 'Processing outcome',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        insurer_reference STRING COMMENT 'Reference to the insurance organization',
        provider_reference STRING COMMENT 'Reference to the billing provider',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        payment_amount DECIMAL(18,2) COMMENT 'Payment monetary amount',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        payment_date DATE COMMENT 'Payment date',
        total_amount DECIMAL(18,2) COMMENT 'Total monetary amount',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        payment_currency STRING COMMENT 'Payment currency code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        use_code STRING COMMENT 'Use code (claim, preauthorization, etc.)',
        total_currency STRING COMMENT 'Total amount currency code',
        system_created_datetime TIMESTAMP COMMENT 'System-level creation timestamp',
        identifier_value STRING COMMENT 'Identifier value within the system',
        claim_reference STRING COMMENT 'Reference to the associated claim',
        type_code STRING COMMENT 'Type code value',
        type_system STRING COMMENT 'Type code system URI',
        system_updated_datetime TIMESTAMP COMMENT 'System-level update timestamp'
    ) TBLPROPERTIES (
        'fhir_resource' = 'ExplanationOfBenefit',
        'domain' = 'Financial',
        'sub_domain' = 'Benefits',
        'description' = 'Explanation of benefit records linking claims to payments including outcome, payment amount, and dates'
    )
    """, SESSION_ID)
    print("Created table: explanation_of_benefit (22 columns)")
except Exception as e:
    print(f"Error creating explanation_of_benefit: {e}")

## Medication Domain (3 tables)

Tables: `medication_administration`, `medication`, `medication_request`

In [ ]:
# MedicationAdministration - Records of medication administration events including dosage, route, effective p
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS medication_administration (
        reason_code_system STRING COMMENT 'Reason code system URI',
        dosage_text STRING COMMENT 'Dosage instruction text',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        dosage_quantity DECIMAL(18,6) COMMENT 'Dosage quantity',
        route_system STRING COMMENT 'Administration route code system URI',
        route_code STRING COMMENT 'Administration route code',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        medication_code_system STRING COMMENT 'Medication code system URI',
        resource_id STRING COMMENT 'Unique resource identifier',
        effective_start_datetime TIMESTAMP COMMENT 'Effective period start datetime',
        medication_reference STRING COMMENT 'Reference to the medication resource',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        effective_end_datetime TIMESTAMP COMMENT 'Effective period end datetime',
        route_display STRING COMMENT 'Administration route display text',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        practitioner_reference STRING COMMENT 'Reference to the practitioner',
        reason_code_value STRING COMMENT 'Reason code value',
        reason_code_display STRING COMMENT 'Reason code display text',
        request_reference STRING COMMENT 'Reference to the originating request',
        context_reference STRING COMMENT 'Reference to the encounter or episode context',
        medication_code_value STRING COMMENT 'Medication code value',
        medication_code_display STRING COMMENT 'Medication code display text',
        dosage_unit STRING COMMENT 'Dosage unit'
    ) TBLPROPERTIES (
        'fhir_resource' = 'MedicationAdministration',
        'domain' = 'Medication',
        'sub_domain' = 'Administration',
        'description' = 'Records of medication administration events including dosage, route, effective period, and reason'
    )
    """, SESSION_ID)
    print("Created table: medication_administration (24 columns)")
except Exception as e:
    print(f"Error creating medication_administration: {e}")

In [ ]:
# Medication - Medication catalog entries including code, form, manufacturer, and amount/streng
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS medication (
        amount_denominator DECIMAL(18,6) COMMENT 'Amount ratio denominator',
        amount_numerator DECIMAL(18,6) COMMENT 'Amount ratio numerator',
        manufacturer_reference STRING COMMENT 'Reference to the manufacturer organization',
        amount_unit STRING COMMENT 'Amount unit',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_id STRING COMMENT 'Unique resource identifier',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        code_text STRING COMMENT 'Free-text description of the code',
        code_system STRING COMMENT 'Code system URI for the primary code',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        form_display STRING COMMENT 'Medication form display text',
        form_code STRING COMMENT 'Medication form code',
        code_value STRING COMMENT 'Primary code value',
        code_display STRING COMMENT 'Display text for the primary code',
        form_system STRING COMMENT 'Medication form code system URI'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Medication',
        'domain' = 'Medication',
        'sub_domain' = 'Catalog',
        'description' = 'Medication catalog entries including code, form, manufacturer, and amount/strength information'
    )
    """, SESSION_ID)
    print("Created table: medication (16 columns)")
except Exception as e:
    print(f"Error creating medication: {e}")

In [ ]:
# MedicationRequest - Medication prescriptions and orders including dosage instructions, route, intent
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS medication_request (
        route_system STRING COMMENT 'Administration route code system URI',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        medication_code_system STRING COMMENT 'Medication code system URI',
        category_display STRING COMMENT 'Category display text',
        resource_id STRING COMMENT 'Unique resource identifier',
        intent STRING COMMENT 'Intent of the resource (proposal, plan, order, etc.)',
        route_display STRING COMMENT 'Administration route display text',
        reason_code_display STRING COMMENT 'Reason code display text',
        medication_code_value STRING COMMENT 'Medication code value',
        dosage_unit STRING COMMENT 'Dosage unit',
        reason_code_system STRING COMMENT 'Reason code system URI',
        dosage_text STRING COMMENT 'Dosage instruction text',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        dosage_quantity DECIMAL(18,6) COMMENT 'Dosage quantity',
        route_code STRING COMMENT 'Administration route code',
        category_code STRING COMMENT 'Category code value',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        authored_datetime TIMESTAMP COMMENT 'Date/time the request was authored',
        medication_reference STRING COMMENT 'Reference to the medication resource',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        dosage_sequence INT COMMENT 'Dosage instruction sequence number',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        reason_code_value STRING COMMENT 'Reason code value',
        category_system STRING COMMENT 'Category code system URI',
        encounter_reference STRING COMMENT 'Reference to the associated encounter',
        request_reference STRING COMMENT 'Reference to the originating request',
        medication_code_display STRING COMMENT 'Medication code display text'
    ) TBLPROPERTIES (
        'fhir_resource' = 'MedicationRequest',
        'domain' = 'Medication',
        'sub_domain' = 'Prescriptions',
        'description' = 'Medication prescriptions and orders including dosage instructions, route, intent, and reason'
    )
    """, SESSION_ID)
    print("Created table: medication_request (27 columns)")
except Exception as e:
    print(f"Error creating medication_request: {e}")

## Security Domain (2 tables)

Tables: `document_reference`, `provenance`

In [ ]:
# DocumentReference - References to clinical documents including content attachment details, category,
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS document_reference (
        content_attachment_hash STRING COMMENT 'Content attachment hash for integrity',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        category_display STRING COMMENT 'Category display text',
        resource_id STRING COMMENT 'Unique resource identifier',
        identifier_system STRING COMMENT 'Identifier coding system URI',
        custodian_reference STRING COMMENT 'Reference to the document custodian',
        date TIMESTAMP COMMENT 'Document date',
        document_status STRING COMMENT 'Document composition status',
        identifier_value STRING COMMENT 'Identifier value within the system',
        content_attachment_type STRING COMMENT 'Content MIME type',
        security_classification STRING COMMENT 'Security classification label',
        type_code STRING COMMENT 'Type code value',
        subject_reference STRING COMMENT 'Reference to the patient (subject)',
        category_code STRING COMMENT 'Category code value',
        type_display STRING COMMENT 'Type display text',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        status STRING COMMENT 'Resource status (active, completed, cancelled, etc.)',
        content_attachment_url STRING COMMENT 'Content attachment URL',
        description STRING COMMENT 'Free-text description',
        category_system STRING COMMENT 'Category code system URI',
        content_attachment_title STRING COMMENT 'Content attachment title',
        context_reference STRING COMMENT 'Reference to the encounter or episode context',
        author_reference STRING COMMENT 'Reference to the document author',
        type_system STRING COMMENT 'Type code system URI',
        content_attachment_size BIGINT COMMENT 'Content attachment size in bytes'
    ) TBLPROPERTIES (
        'fhir_resource' = 'DocumentReference',
        'domain' = 'Security',
        'sub_domain' = 'Documents',
        'description' = 'References to clinical documents including content attachment details, category, security classification, and authorship'
    )
    """, SESSION_ID)
    print("Created table: document_reference (26 columns)")
except Exception as e:
    print(f"Error creating document_reference: {e}")

In [ ]:
# Provenance - Provenance records tracking data origin and changes including agent, activity, r
try:
    run_sql("""
    CREATE TABLE IF NOT EXISTS provenance (
        activity_code STRING COMMENT 'Provenance activity code',
        agent_type_system STRING COMMENT 'Agent type code system URI',
        agent_type_code STRING COMMENT 'Agent type code',
        reason_code STRING COMMENT 'Reason code value',
        reason_system STRING COMMENT 'Reason code system URI',
        agent_on_behalf_of_reference STRING COMMENT 'Reference to the entity the agent acted on behalf of',
        updated_datetime TIMESTAMP COMMENT 'Record last update timestamp',
        occurrence_end_datetime TIMESTAMP COMMENT 'Occurrence period end datetime',
        created_datetime TIMESTAMP COMMENT 'Record creation timestamp',
        occurrence_start_datetime TIMESTAMP COMMENT 'Occurrence period start datetime',
        recorded_date TIMESTAMP COMMENT 'Date the resource was recorded',
        resource_id STRING COMMENT 'Unique resource identifier',
        agent_who_reference STRING COMMENT 'Reference to the provenance agent',
        resource_type STRING COMMENT 'FHIR resource type discriminator',
        activity_display STRING COMMENT 'Provenance activity display text',
        target_reference STRING COMMENT 'Reference to the target resource',
        reason_display STRING COMMENT 'Reason display text',
        agent_type_display STRING COMMENT 'Agent type display text',
        activity_system STRING COMMENT 'Provenance activity code system URI',
        location_reference STRING COMMENT 'Reference to the location'
    ) TBLPROPERTIES (
        'fhir_resource' = 'Provenance',
        'domain' = 'Security',
        'sub_domain' = 'Audit',
        'description' = 'Provenance records tracking data origin and changes including agent, activity, reason, and target references'
    )
    """, SESSION_ID)
    print("Created table: provenance (20 columns)")
except Exception as e:
    print(f"Error creating provenance: {e}")

## Verification

## Column Mapping Reference

Maps original abbreviated column names to expanded names for data loading.

## Data Loading Example (Template)

## Step 3: 데이터 로딩 (Aurora PostgreSQL → S3 Tables)

# FHIR Data Loading: Aurora PostgreSQL to S3 Tables

Loads 24 tables (498 columns) from Aurora PostgreSQL into S3 Tables.

Each table is read via JDBC, columns are renamed from abbreviated to expanded names, and data is inserted into the corresponding S3 Table.

In [ ]:
# JDBC 설정 (세션 초기화 시 이미 설정됨)
import builtins
jdbc_url = getattr(builtins, '_jdbc_url', None)
creds = getattr(builtins, '_jdbc_creds', None)
driver_path = getattr(builtins, '_driver_path', None)

if not jdbc_url:
    import boto3, json
    session = boto3.session.Session()
    region = session.region_name
    account_id = session.client('sts').get_caller_identity()['Account']
    secret_id = boto3.client('secretsmanager', region_name=region).list_secrets()['SecretList'][0]['Name']
    creds = json.loads(boto3.client('secretsmanager', region_name=region).get_secret_value(SecretId=secret_id)['SecretString'])
    jdbc_url = f"jdbc:postgresql://{creds['host']}:{creds['port']}/{creds['dbname']}"
    driver_path = f"s3://fhir-data-{account_id}-{region}/lib/postgresql-42.7.1.jar"

print(f"JDBC URL: {jdbc_url}")


In [ ]:
# JDBC 설정 (세션 초기화 시 이미 설정됨)
import builtins
jdbc_url = getattr(builtins, '_jdbc_url', None)
creds = getattr(builtins, '_jdbc_creds', None)
driver_path = getattr(builtins, '_driver_path', None)

if not jdbc_url:
    import boto3, json
    session = boto3.session.Session()
    region = session.region_name
    account_id = session.client('sts').get_caller_identity()['Account']
    secret_id = boto3.client('secretsmanager', region_name=region).list_secrets()['SecretList'][0]['Name']
    creds = json.loads(boto3.client('secretsmanager', region_name=region).get_secret_value(SecretId=secret_id)['SecretString'])
    jdbc_url = f"jdbc:postgresql://{creds['host']}:{creds['port']}/{creds['dbname']}"
    driver_path = f"s3://fhir-data-{account_id}-{region}/lib/postgresql-42.7.1.jar"

print(f"JDBC URL: {jdbc_url}")


## Administrative Domain (5 tables)

In [ ]:
pyspark_code = """
# Load patient (Patient)
try:
    df = read_source_table("public.abc_reg_ptnt")
    df.createOrReplaceTempView("source_patient")
    spark.sql("""
    INSERT INTO patient
    SELECT
        addr_cty AS address_city,
        CAST(mlt_bth_ind AS BOOLEAN) AS multiple_birth_indicator,
        gndr AS gender,
        tlc_sys AS telecom_system,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        addr_zip AS address_postal_code,
        addr_ctr AS address_country,
        rid AS resource_id,
        idn_sys AS identifier_system,
        dcd_dt AS deceased_datetime,
        rtp AS resource_type,
        lng_cd AS language_code,
        sts AS status,
        nm_fam AS name_family,
        tlc_val AS telecom_value,
        CAST(bth_dt AS DATE) AS birth_date,
        idn_val AS identifier_value,
        addr_ln1 AS address_line_1,
        addr_st AS address_state,
        mlt_bth_num AS multiple_birth_number,
        addr_ln2 AS address_line_2,
        mrt_sts AS marital_status,
        nm_gvn AS name_given
    FROM source_patient
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM patient").collect()[0]["cnt"]
    print(f"Loaded patient: {count} rows")
except Exception as e:
    print(f"Error loading patient: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load practitioner (Practitioner)
try:
    df = read_source_table("public.abc_reg_prct")
    df.createOrReplaceTempView("source_practitioner")
    spark.sql("""
    INSERT INTO practitioner
    SELECT
        addr_cty AS address_city,
        nm_pfx AS name_prefix,
        gndr AS gender,
        nm_sfx AS name_suffix,
        tlc_sys AS telecom_system,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        addr_zip AS address_postal_code,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        nm_fam AS name_family,
        tlc_val AS telecom_value,
        CAST(bth_dt AS DATE) AS birth_date,
        idn_val AS identifier_value,
        addr_ln1 AS address_line_1,
        addr_st AS address_state,
        CAST(actv_ind AS BOOLEAN) AS active_indicator,
        nm_gvn AS name_given
    FROM source_practitioner
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM practitioner").collect()[0]["cnt"]
    print(f"Loaded practitioner: {count} rows")
except Exception as e:
    print(f"Error loading practitioner: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load organization (Organization)
try:
    df = read_source_table("public.abc_reg_orgz")
    df.createOrReplaceTempView("source_organization")
    spark.sql("""
    INSERT INTO organization
    SELECT
        addr_cty AS address_city,
        tlc_sys AS telecom_system,
        upd_dts AS updated_datetime,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        addr_zip AS address_postal_code,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        tlc_val AS telecom_value,
        idn_val AS identifier_value,
        addr_ln1 AS address_line_1,
        addr_st AS address_state,
        typ_cd AS type_code,
        CAST(actv_ind AS BOOLEAN) AS active_indicator,
        nm AS name
    FROM source_organization
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM organization").collect()[0]["cnt"]
    print(f"Loaded organization: {count} rows")
except Exception as e:
    print(f"Error loading organization: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load location (Location)
try:
    df = read_source_table("public.abc_reg_lctn")
    df.createOrReplaceTempView("source_location")
    spark.sql("""
    INSERT INTO location
    SELECT
        addr_cty AS address_city,
        mng_org_ref AS managing_organization_reference,
        tlc_sys AS telecom_system,
        upd_dts AS updated_datetime,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        addr_zip AS address_postal_code,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        CAST(pos_lat AS DECIMAL(10,7)) AS position_latitude,
        sts AS status,
        tlc_val AS telecom_value,
        dsc AS description,
        idn_val AS identifier_value,
        addr_ln1 AS address_line_1,
        addr_st AS address_state,
        modl AS mode,
        typ_cd AS type_code,
        nm AS name,
        CAST(pos_lng AS DECIMAL(10,7)) AS position_longitude
    FROM source_location
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM location").collect()[0]["cnt"]
    print(f"Loaded location: {count} rows")
except Exception as e:
    print(f"Error loading location: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load practitioner_role (PractitionerRole)
try:
    df = read_source_table("public.abc_reg_prol")
    df.createOrReplaceTempView("source_practitioner_role")
    spark.sql("""
    INSERT INTO practitioner_role
    SELECT
        spc_val AS specialty_value,
        spc_dsp AS specialty_display,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        rid AS resource_id,
        idn_sys AS identifier_system,
        org_ref AS organization_reference,
        rtp AS resource_type,
        cd_sys AS code_system,
        prf_ref AS practitioner_reference,
        idn_val AS identifier_value,
        CAST(actv_ind AS BOOLEAN) AS active_indicator,
        cd_val AS code_value,
        cd_dsp AS code_display,
        loc_ref AS location_reference,
        spc_sys AS specialty_system
    FROM source_practitioner_role
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM practitioner_role").collect()[0]["cnt"]
    print(f"Loaded practitioner_role: {count} rows")
except Exception as e:
    print(f"Error loading practitioner_role: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


## Clinical Domain (12 tables)

In [ ]:
pyspark_code = """
# Load encounter (Encounter)
try:
    df = read_source_table("public.abc_cln_enct")
    df.createOrReplaceTempView("source_encounter")
    spark.sql("""
    INSERT INTO encounter
    SELECT
        rsn_cd_sys AS reason_code_system,
        cls_cd AS class_code,
        sbj_ref AS subject_reference,
        prd_ed_dts AS period_end_datetime,
        upd_dts AS updated_datetime,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        rid AS resource_id,
        idn_sys AS identifier_system,
        org_ref AS organization_reference,
        cls_sys AS class_system,
        rtp AS resource_type,
        prd_st_dts AS period_start_datetime,
        svc_prf_ref AS service_provider_reference,
        sts AS status,
        rsn_cd_val AS reason_code_value,
        rsn_cd_dsp AS reason_code_display,
        idn_val AS identifier_value,
        typ_cd AS type_code,
        typ_sys AS type_system,
        loc_ref AS location_reference
    FROM source_encounter
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM encounter").collect()[0]["cnt"]
    print(f"Loaded encounter: {count} rows")
except Exception as e:
    print(f"Error loading encounter: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load condition (Condition)
try:
    df = read_source_table("public.abc_cln_cond")
    df.createOrReplaceTempView("source_condition")
    spark.sql("""
    INSERT INTO condition
    SELECT
        cln_sts_sys AS clinical_status_system,
        abd_dts AS abatement_datetime,
        cln_sts_cd AS clinical_status_code,
        sbj_ref AS subject_reference,
        ctg_cd AS category_code,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        ctg_dsp AS category_display,
        CAST(rec_dts AS DATE) AS recorded_date,
        rid AS resource_id,
        rtp AS resource_type,
        cd_txt AS code_text,
        vrf_sts_cd AS verification_status_code,
        cd_sys AS code_system,
        ons_dts AS onset_datetime,
        ctg_sys AS category_system,
        evt_ref AS encounter_reference,
        vrf_sts_sys AS verification_status_system,
        cd_val AS code_value,
        cd_dsp AS code_display
    FROM source_condition
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM condition").collect()[0]["cnt"]
    print(f"Loaded condition: {count} rows")
except Exception as e:
    print(f"Error loading condition: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load observation (Observation)
try:
    df = read_source_table("public.abc_cln_obsv")
    df.createOrReplaceTempView("source_observation")
    spark.sql("""
    INSERT INTO observation
    SELECT
        iss_dts AS issued_datetime,
        itp_txt AS interpretation_text,
        eff_dts AS effective_datetime,
        sbj_ref AS subject_reference,
        ctg_cd AS category_code,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        ctg_dsp AS category_display,
        val_sys AS value_system,
        rid AS resource_id,
        rtp AS resource_type,
        cd_txt AS code_text,
        cd_sys AS code_system,
        sts AS status,
        CAST(val_bool AS BOOLEAN) AS value_boolean,
        ctg_sys AS category_system,
        evt_ref AS encounter_reference,
        val_str AS value_string,
        val_unt AS value_unit,
        val_cd AS value_code,
        CAST(val_qty AS DECIMAL(18,6)) AS value_quantity,
        cd_val AS code_value,
        cd_dsp AS code_display
    FROM source_observation
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM observation").collect()[0]["cnt"]
    print(f"Loaded observation: {count} rows")
except Exception as e:
    print(f"Error loading observation: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load procedure (Procedure)
try:
    df = read_source_table("public.abc_cln_prcd")
    df.createOrReplaceTempView("source_procedure")
    spark.sql("""
    INSERT INTO procedure
    SELECT
        sbj_ref AS subject_reference,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        rid AS resource_id,
        rtp AS resource_type,
        cd_txt AS code_text,
        prf_st_dts AS performed_start_datetime,
        cd_sys AS code_system,
        sts AS status,
        prf_ed_dts AS performed_end_datetime,
        prf_ref AS practitioner_reference,
        evt_ref AS encounter_reference,
        cd_val AS code_value,
        cd_dsp AS code_display,
        loc_ref AS location_reference,
        rsn_ref AS reason_reference
    FROM source_procedure
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM procedure").collect()[0]["cnt"]
    print(f"Loaded procedure: {count} rows")
except Exception as e:
    print(f"Error loading procedure: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load allergy_intolerance (AllergyIntolerance)
try:
    df = read_source_table("public.abc_dgn_algy")
    df.createOrReplaceTempView("source_allergy_intolerance")
    spark.sql("""
    INSERT INTO allergy_intolerance
    SELECT
        cln_sts_sys AS clinical_status_system,
        cln_sts_cd AS clinical_status_code,
        crt AS criticality,
        sbj_ref AS subject_reference,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        typ AS type,
        rec_dts AS recorded_date,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        vrf_sts_cd AS verification_status_code,
        cd_sys AS code_system,
        ons_dts AS onset_datetime,
        ctg AS category,
        evt_ref AS encounter_reference,
        idn_val AS identifier_value,
        vrf_sts_sys AS verification_status_system,
        rec_ref AS recorder_reference,
        cd_val AS code_value,
        cd_dsp AS code_display
    FROM source_allergy_intolerance
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM allergy_intolerance").collect()[0]["cnt"]
    print(f"Loaded allergy_intolerance: {count} rows")
except Exception as e:
    print(f"Error loading allergy_intolerance: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load diagnostic_report (DiagnosticReport)
try:
    df = read_source_table("public.abc_dgn_drpt")
    df.createOrReplaceTempView("source_diagnostic_report")
    spark.sql("""
    INSERT INTO diagnostic_report
    SELECT
        iss_dts AS issued_datetime,
        eff_dts AS effective_datetime,
        sbj_ref AS subject_reference,
        ctg_cd AS category_code,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        ctg_dsp AS category_display,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        cd_txt AS code_text,
        cd_sys AS code_system,
        sts AS status,
        prf_ref AS practitioner_reference,
        ctg_sys AS category_system,
        evt_ref AS encounter_reference,
        cnc_txt AS conclusion_text,
        idn_val AS identifier_value,
        cd_val AS code_value,
        cd_dsp AS code_display
    FROM source_diagnostic_report
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM diagnostic_report").collect()[0]["cnt"]
    print(f"Loaded diagnostic_report: {count} rows")
except Exception as e:
    print(f"Error loading diagnostic_report: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load imaging_study (ImagingStudy)
try:
    df = read_source_table("public.abc_dgn_imgs")
    df.createOrReplaceTempView("source_imaging_study")
    spark.sql("""
    INSERT INTO imaging_study
    SELECT
        rsn_cd_sys AS reason_code_system,
        sbj_ref AS subject_reference,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        std_dts AS started_datetime,
        rid AS resource_id,
        idn_sys AS identifier_system,
        itp_ref AS interpreter_reference,
        rtp AS resource_type,
        modl_sys AS modality_system,
        prc_cd_sys AS procedure_code_system,
        modl_cd AS modality_code,
        sts AS status,
        num_ins AS number_of_instances,
        rsn_cd_val AS reason_code_value,
        rsn_cd_dsp AS reason_code_display,
        dsc AS description,
        evt_ref AS encounter_reference,
        idn_val AS identifier_value,
        rfr_ref AS referrer_reference,
        num_srs AS number_of_series,
        modl_dsp AS modality_display,
        prc_cd_val AS procedure_code_value,
        prc_cd_dsp AS procedure_code_display
    FROM source_imaging_study
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM imaging_study").collect()[0]["cnt"]
    print(f"Loaded imaging_study: {count} rows")
except Exception as e:
    print(f"Error loading imaging_study: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load immunization (Immunization)
try:
    df = read_source_table("public.abc_dgn_imzn")
    df.createOrReplaceTempView("source_immunization")
    spark.sql("""
    INSERT INTO immunization
    SELECT
        vac_cd_dsp AS vaccine_code_display,
        sts_rsn_sys AS status_reason_system,
        lot_num AS lot_number,
        CAST(pry_src AS BOOLEAN) AS primary_source,
        mfr_ref AS manufacturer_reference,
        rte_sys AS route_system,
        upd_dts AS updated_datetime,
        rid AS resource_id,
        sts_rsn_dsp AS status_reason_display,
        occ_dts AS occurrence_datetime,
        rte_dsp AS route_display,
        ste_cd AS site_code,
        CAST(exp_dt AS DATE) AS expiration_date,
        vac_cd_sys AS vaccine_code_system,
        dos_unt AS dosage_unit,
        vac_cd_val AS vaccine_code_value,
        sbj_ref AS subject_reference,
        CAST(dos_qty AS DECIMAL(18,6)) AS dosage_quantity,
        sts_rsn_cd AS status_reason_code,
        rte_cd AS route_code,
        ste_sys AS site_system,
        crt_dts AS created_datetime,
        rtp AS resource_type,
        sts AS status,
        prf_ref AS practitioner_reference,
        evt_ref AS encounter_reference,
        ste_dsp AS site_display,
        loc_ref AS location_reference
    FROM source_immunization
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM immunization").collect()[0]["cnt"]
    print(f"Loaded immunization: {count} rows")
except Exception as e:
    print(f"Error loading immunization: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load care_plan (CarePlan)
try:
    df = read_source_table("public.abc_car_cpln")
    df.createOrReplaceTempView("source_care_plan")
    spark.sql("""
    INSERT INTO care_plan
    SELECT
        sbj_ref AS subject_reference,
        prd_ed_dts AS period_end_datetime,
        ctg_cd AS category_code,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        ctg_dsp AS category_display,
        rid AS resource_id,
        idn_sys AS identifier_system,
        ttl AS title,
        intnt AS intent,
        rtp AS resource_type,
        prd_st_dts AS period_start_datetime,
        sts AS status,
        dsc AS description,
        ctg_sys AS category_system,
        evt_ref AS encounter_reference,
        idn_val AS identifier_value
    FROM source_care_plan
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM care_plan").collect()[0]["cnt"]
    print(f"Loaded care_plan: {count} rows")
except Exception as e:
    print(f"Error loading care_plan: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load care_team (CareTeam)
try:
    df = read_source_table("public.abc_car_ctm")
    df.createOrReplaceTempView("source_care_team")
    spark.sql("""
    INSERT INTO care_team
    SELECT
        mng_org_ref AS managing_organization_reference,
        sbj_ref AS subject_reference,
        prd_ed_dts AS period_end_datetime,
        ctg_cd AS category_code,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        ctg_dsp AS category_display,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        prd_st_dts AS period_start_datetime,
        sts AS status,
        ctg_sys AS category_system,
        evt_ref AS encounter_reference,
        idn_val AS identifier_value,
        nm AS name
    FROM source_care_team
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM care_team").collect()[0]["cnt"]
    print(f"Loaded care_team: {count} rows")
except Exception as e:
    print(f"Error loading care_team: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load device (Device)
try:
    df = read_source_table("public.abc_car_devc")
    df.createOrReplaceTempView("source_device")
    spark.sql("""
    INSERT INTO device
    SELECT
        lot_num AS lot_number,
        sbj_ref AS subject_reference,
        srl_num AS serial_number,
        upd_dts AS updated_datetime,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        sts AS status,
        mdl_num AS model_number,
        vrs AS version,
        idn_val AS identifier_value,
        typ_txt AS type_text,
        mfr AS manufacturer,
        typ_cd AS type_code,
        CAST(exp_dt AS DATE) AS expiration_date,
        typ_sys AS type_system,
        loc_ref AS location_reference
    FROM source_device
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM device").collect()[0]["cnt"]
    print(f"Loaded device: {count} rows")
except Exception as e:
    print(f"Error loading device: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load supply_delivery (SupplyDelivery)
try:
    df = read_source_table("public.abc_car_sdlv")
    df.createOrReplaceTempView("source_supply_delivery")
    spark.sql("""
    INSERT INTO supply_delivery
    SELECT
        sbj_ref AS subject_reference,
        dst_ref AS destination_reference,
        sup_ref AS supplier_reference,
        upd_dts AS updated_datetime,
        typ_dsp AS type_display,
        occ_ed_dts AS occurrence_end_datetime,
        crt_dts AS created_datetime,
        occ_st_dts AS occurrence_start_datetime,
        rid AS resource_id,
        idn_sys AS identifier_system,
        rtp AS resource_type,
        sts AS status,
        sup_itm_cd_sys AS supplied_item_code_system,
        CAST(qty AS DECIMAL(18,6)) AS quantity,
        idn_val AS identifier_value,
        sup_itm_cd_val AS supplied_item_code_value,
        sup_itm_cd_dsp AS supplied_item_code_display,
        typ_cd AS type_code,
        qty_unt AS quantity_unit,
        typ_sys AS type_system,
        sup_itm_ref AS supplied_item_reference
    FROM source_supply_delivery
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM supply_delivery").collect()[0]["cnt"]
    print(f"Loaded supply_delivery: {count} rows")
except Exception as e:
    print(f"Error loading supply_delivery: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


## Financial Domain (2 tables)

In [ ]:
pyspark_code = """
# Load claim (Claim)
try:
    df = read_source_table("public.abc_fin_clam")
    df.createOrReplaceTempView("source_claim")
    spark.sql("""
    INSERT INTO claim
    SELECT
        sbj_ref AS subject_reference,
        ins_ref AS insurer_reference,
        prv_ref AS provider_reference,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        rid AS resource_id,
        idn_sys AS identifier_system,
        CAST(tot_amt AS DECIMAL(18,2)) AS total_amount,
        rtp AS resource_type,
        prs_cd_sys AS priority_code_system,
        sts AS status,
        use_cd AS use_code,
        pry AS priority,
        tot_cur AS total_currency,
        sys_crt_dts AS system_created_datetime,
        idn_val AS identifier_value,
        typ_cd AS type_code,
        typ_sys AS type_system,
        sys_upd_dts AS system_updated_datetime,
        prs_cd_val AS priority_code_value,
        prs_cd_dsp AS priority_code_display
    FROM source_claim
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM claim").collect()[0]["cnt"]
    print(f"Loaded claim: {count} rows")
except Exception as e:
    print(f"Error loading claim: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load explanation_of_benefit (ExplanationOfBenefit)
try:
    df = read_source_table("public.abc_fin_eob")
    df.createOrReplaceTempView("source_explanation_of_benefit")
    spark.sql("""
    INSERT INTO explanation_of_benefit
    SELECT
        outc AS outcome,
        sbj_ref AS subject_reference,
        ins_ref AS insurer_reference,
        prv_ref AS provider_reference,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        CAST(pmt_amt AS DECIMAL(18,2)) AS payment_amount,
        rid AS resource_id,
        idn_sys AS identifier_system,
        CAST(pmt_dt AS DATE) AS payment_date,
        CAST(tot_amt AS DECIMAL(18,2)) AS total_amount,
        rtp AS resource_type,
        pmt_cur AS payment_currency,
        sts AS status,
        use_cd AS use_code,
        tot_cur AS total_currency,
        sys_crt_dts AS system_created_datetime,
        idn_val AS identifier_value,
        clm_ref AS claim_reference,
        typ_cd AS type_code,
        typ_sys AS type_system,
        sys_upd_dts AS system_updated_datetime
    FROM source_explanation_of_benefit
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM explanation_of_benefit").collect()[0]["cnt"]
    print(f"Loaded explanation_of_benefit: {count} rows")
except Exception as e:
    print(f"Error loading explanation_of_benefit: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


## Medication Domain (3 tables)

In [ ]:
pyspark_code = """
# Load medication_administration (MedicationAdministration)
try:
    df = read_source_table("public.abc_rx_madm")
    df.createOrReplaceTempView("source_medication_administration")
    spark.sql("""
    INSERT INTO medication_administration
    SELECT
        rsn_cd_sys AS reason_code_system,
        dos_txt AS dosage_text,
        sbj_ref AS subject_reference,
        CAST(dos_qty AS DECIMAL(18,6)) AS dosage_quantity,
        rte_sys AS route_system,
        rte_cd AS route_code,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        med_cd_sys AS medication_code_system,
        rid AS resource_id,
        eff_st_dts AS effective_start_datetime,
        med_ref AS medication_reference,
        rtp AS resource_type,
        eff_ed_dts AS effective_end_datetime,
        rte_dsp AS route_display,
        sts AS status,
        prf_ref AS practitioner_reference,
        rsn_cd_val AS reason_code_value,
        rsn_cd_dsp AS reason_code_display,
        req_ref AS request_reference,
        ctx_ref AS context_reference,
        med_cd_val AS medication_code_value,
        med_cd_dsp AS medication_code_display,
        dos_unt AS dosage_unit
    FROM source_medication_administration
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM medication_administration").collect()[0]["cnt"]
    print(f"Loaded medication_administration: {count} rows")
except Exception as e:
    print(f"Error loading medication_administration: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load medication (Medication)
try:
    df = read_source_table("public.abc_rx_mdcn")
    df.createOrReplaceTempView("source_medication")
    spark.sql("""
    INSERT INTO medication
    SELECT
        CAST(amt_dnm AS DECIMAL(18,6)) AS amount_denominator,
        CAST(amt_num AS DECIMAL(18,6)) AS amount_numerator,
        mfr_ref AS manufacturer_reference,
        amt_unt AS amount_unit,
        upd_dts AS updated_datetime,
        crt_dts AS created_datetime,
        rid AS resource_id,
        rtp AS resource_type,
        cd_txt AS code_text,
        cd_sys AS code_system,
        sts AS status,
        frm_dsp AS form_display,
        frm_cd AS form_code,
        cd_val AS code_value,
        cd_dsp AS code_display,
        frm_sys AS form_system
    FROM source_medication
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM medication").collect()[0]["cnt"]
    print(f"Loaded medication: {count} rows")
except Exception as e:
    print(f"Error loading medication: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load medication_request (MedicationRequest)
try:
    df = read_source_table("public.abc_rx_mreq")
    df.createOrReplaceTempView("source_medication_request")
    spark.sql("""
    INSERT INTO medication_request
    SELECT
        rte_sys AS route_system,
        upd_dts AS updated_datetime,
        med_cd_sys AS medication_code_system,
        ctg_dsp AS category_display,
        rid AS resource_id,
        intnt AS intent,
        rte_dsp AS route_display,
        rsn_cd_dsp AS reason_code_display,
        med_cd_val AS medication_code_value,
        dos_unt AS dosage_unit,
        rsn_cd_sys AS reason_code_system,
        dos_txt AS dosage_text,
        sbj_ref AS subject_reference,
        CAST(dos_qty AS DECIMAL(18,6)) AS dosage_quantity,
        rte_cd AS route_code,
        ctg_cd AS category_code,
        crt_dts AS created_datetime,
        ath_dts AS authored_datetime,
        med_ref AS medication_reference,
        rtp AS resource_type,
        dos_seq AS dosage_sequence,
        sts AS status,
        rsn_cd_val AS reason_code_value,
        ctg_sys AS category_system,
        evt_ref AS encounter_reference,
        req_ref AS request_reference,
        med_cd_dsp AS medication_code_display
    FROM source_medication_request
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM medication_request").collect()[0]["cnt"]
    print(f"Loaded medication_request: {count} rows")
except Exception as e:
    print(f"Error loading medication_request: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


## Security Domain (2 tables)

In [ ]:
pyspark_code = """
# Load document_reference (DocumentReference)
try:
    df = read_source_table("public.abc_doc_dref")
    df.createOrReplaceTempView("source_document_reference")
    spark.sql("""
    INSERT INTO document_reference
    SELECT
        cnt_att_hsh AS content_attachment_hash,
        upd_dts AS updated_datetime,
        ctg_dsp AS category_display,
        rid AS resource_id,
        idn_sys AS identifier_system,
        cst_ref AS custodian_reference,
        dt AS date,
        doc_sts AS document_status,
        idn_val AS identifier_value,
        cnt_att_typ AS content_attachment_type,
        sec_cls AS security_classification,
        typ_cd AS type_code,
        sbj_ref AS subject_reference,
        ctg_cd AS category_code,
        typ_dsp AS type_display,
        crt_dts AS created_datetime,
        rtp AS resource_type,
        sts AS status,
        cnt_att_url AS content_attachment_url,
        dsc AS description,
        ctg_sys AS category_system,
        cnt_att_ttl AS content_attachment_title,
        ctx_ref AS context_reference,
        ath_ref AS author_reference,
        typ_sys AS type_system,
        cnt_att_sz AS content_attachment_size
    FROM source_document_reference
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM document_reference").collect()[0]["cnt"]
    print(f"Loaded document_reference: {count} rows")
except Exception as e:
    print(f"Error loading document_reference: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


In [ ]:
pyspark_code = """
# Load provenance (Provenance)
try:
    df = read_source_table("public.abc_doc_prvn")
    df.createOrReplaceTempView("source_provenance")
    spark.sql("""
    INSERT INTO provenance
    SELECT
        act_cd AS activity_code,
        agt_typ_sys AS agent_type_system,
        agt_typ_cd AS agent_type_code,
        rsn_cd AS reason_code,
        rsn_sys AS reason_system,
        agt_bhf_ref AS agent_on_behalf_of_reference,
        upd_dts AS updated_datetime,
        occ_ed_dts AS occurrence_end_datetime,
        crt_dts AS created_datetime,
        occ_st_dts AS occurrence_start_datetime,
        rec_dts AS recorded_date,
        rid AS resource_id,
        agt_who_ref AS agent_who_reference,
        rtp AS resource_type,
        act_dsp AS activity_display,
        tgt_ref AS target_reference,
        rsn_dsp AS reason_display,
        agt_typ_dsp AS agent_type_display,
        act_sys AS activity_system,
        loc_ref AS location_reference
    FROM source_provenance
    """)
    count = spark.sql("SELECT COUNT(*) AS cnt FROM provenance").collect()[0]["cnt"]
    print(f"Loaded provenance: {count} rows")
except Exception as e:
    print(f"Error loading provenance: {e}")
"""
result = run_pyspark(pyspark_code, SESSION_ID)
if result:
    print(result)


## Verification: Row Counts

## Sample Data

## Step 4: 검증

In [ ]:
# 각 테이블 row count 확인
tables = [
    "patient", "practitioner", "organization", "location", "practitioner_role",
    "encounter", "condition", "observation", "procedure", "allergy_intolerance",
    "diagnostic_report", "imaging_study", "immunization", "care_plan", "care_team",
    "device", "medication", "medication_request", "medication_administration",
    "claim", "explanation_of_benefit", "document_reference", "provenance", "supply_delivery"
]

print("=== Row Counts ===")
total = 0
for t in tables:
    result = run_sql(f"SELECT COUNT(*) as cnt FROM {t}", SESSION_ID)
    cnt = result.get('application/json', [{}])[0].get('cnt', 0) if result else 0
    print(f"  {t}: {cnt:,}")
    total += int(cnt)
print(f"\nTotal: {total:,} rows")
